# Supplementary Figure 8

**Comparison of germinated and ungerminated counts when counting manually or with Mycol assistance**

- **Data** — three raters scored the same 120 images twice, by hand and with Mycol
- **Does** — per rater pair, the mean absolute count difference per image, germinated and
  ungerminated separately; error bars are 95% CI (1.96 x SEM)
- **See also** — `analyses/mycol_vs_manual.ipynb` for ICC, per-image variance and the timing behind
  Figure 3f
- **Kernel** — `mycol_colonies_env`, run top to bottom


## The rater-pair differences


In [ ]:
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from PIL import Image

OUTPUT = Path("output")
OUTPUT.mkdir(exist_ok=True)
XLSX = "assets/20250908_All_data_manual_vs_tool.xlsx"

# One sheet per rater x method. The sheets are named after the raters; they are
# relabelled User1/2/3 here, and disagree on column names ("Photo" vs "Image",
# mixed case, trailing spaces), so the columns are normalised on the way in.
SHEETS = {("User1", "manual"): "Alex_manual", ("User2", "manual"): "Gio_manual",
          ("User3", "manual"): "Rafa_manual", ("User1", "tool"): "Alex_tool",
          ("User2", "tool"): "Gio_tool", ("User3", "tool"): "Rafa_tool"}


def read_sheet(sheet, user, method):
    df = pd.read_excel(XLSX, sheet_name=sheet)
    df.columns = df.columns.str.strip().str.lower()
    df = df.rename(columns={"photo": "image", "number": "image_number"})
    return df[["image_number", "germinated", "ungerminated"]].assign(user=user, method=method)


combined = pd.concat([read_sheet(sheet, u, m) for (u, m), sheet in SHEETS.items()],
                     ignore_index=True)

rows = []
for method, label in [("manual", "Manual"), ("tool", "Mycol")]:
    for count in ["germinated", "ungerminated"]:
        # one column per rater, so the two counts for an image line up
        pivot = combined[combined["method"] == method].pivot_table(
            index="image_number", columns="user", values=count)
        for u1, u2 in combinations(["User1", "User2", "User3"], 2):
            diffs = (pivot[u1] - pivot[u2]).abs()
            rows.append({"Method": label, "Count": count.capitalize(),
                         "Rater pair": f"{u1} \u2013 {u2}",
                         "Mean absolute difference": diffs.mean(),
                         "ci_half": 1.96 * diffs.std() / np.sqrt(diffs.count())})

mad = pd.DataFrame(rows)
print(f"{len(combined)} rows: {combined['image_number'].nunique()} images x 3 raters x 2 methods")
print(mad.to_string(index=False))


## Draw


In [ ]:
fig = px.bar(mad, x="Rater pair", y="Mean absolute difference", error_y="ci_half",
             color="Method", facet_col="Count", barmode="group",
             color_discrete_map={"Manual": "#a8d5a2", "Mycol": "#b8a9d9"},
             labels={"Mean absolute difference": "Mean absolute difference (cells)"},
             width=780, height=430, template="simple_white")
fig.update_traces(marker_line_color="black", marker_line_width=1)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))   # drop "Count="
fig.update_layout(font=dict(size=14))

# 6.25 in square. Laying the figure out in points (inches x 72) makes a font.size
# of 14 render as 14 pt; scale = 300/72 then lifts it to 300 dpi.
out = OUTPUT / "Figure_S8.png"
go.Figure(fig).write_image(out, width=6.25 * 72, height=6.25 * 72, scale=300 / 72)
with Image.open(out) as im:
    im.save(out, dpi=(300, 300))
print(f"wrote {out}  {out.stat().st_size:,} B")
fig.show()
